# Mamba-3 Compatibility Test

**Goal:** Test if real Mamba-3 (from `state-spaces/mamba` main branch) can run on Colab A100.

| Step | What | Pass Criteria |
|------|------|---------------|
| 1 | Install from main branch | `from mamba_ssm import Mamba3` works |
| 2 | Forward pass | Output shape matches input |
| 3 | Backward pass | `.backward()` completes without crash |
| 4 | CrossScan integration | 3-axis scan with Mamba3 blocks |
| 5 | TextMamba3D smoke test | Full model forward+backward |

**Mamba-3 paper:** arxiv 2603.15569 (ICLR 2026)  
**Dependencies:** triton>=3.5.0, tilelang>=0.1.7.post3, nvidia-cutlass-dsl==4.4.1, quack-kernels==0.3.1

In [1]:
# Cell 1: Environment check
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

import sys
print(f'Python: {sys.version}')

# Check current triton version
try:
    import triton
    print(f'Triton: {triton.__version__}')
except ImportError:
    print('Triton: not installed')

Fri Mar 20 09:36:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             52W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [11]:
# Cell 2: Install Mamba-3 from source

import sys, os, shutil

# 1. Install runtime deps
!pip install -q "triton>=3.5.0" "tilelang>=0.1.7.post3" "nvidia-cutlass-dsl==4.4.1" "quack-kernels==0.3.1" "causal-conv1d>=1.4.0" cuda-python einops transformers

# 2. Clone main branch
SRC = "/content/mamba_src"
if os.path.isdir(SRC):
    shutil.rmtree(SRC)
!git clone --depth 1 https://github.com/state-spaces/mamba.git {SRC}

assert os.path.exists(f"{SRC}/mamba_ssm/modules/mamba3.py")
print(f"mamba3.py found")

# 3. Stub out cute/CUTLASS step fn (inference-only, not needed for training)
stub_path = f"{SRC}/mamba_ssm/ops/cute/mamba3/mamba3_step_fn.py"
if os.path.exists(stub_path):
    shutil.copy2(stub_path, stub_path + ".bak")
    STUB_CODE = '''
# Stubbed - cute/CUTLASS step fn not needed for training
def mamba3_step_fn(*args, **kwargs):
    raise NotImplementedError('mamba3_step_fn requires CUTLASS DSL')
'''
    with open(stub_path, "w") as f:
        f.write(STUB_CODE)
    print("Patched: mamba3_step_fn stubbed")

# 4. sys.path
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# 5. Flush cached mamba_ssm
for key in list(sys.modules.keys()):
    if "mamba_ssm" in key:
        del sys.modules[key]

# 6. Import
try:
    from mamba_ssm import Mamba, Mamba2, Mamba3
    print(f"Mamba3: {Mamba3}")
    print("OK")
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()


Cloning into '/content/mamba_src'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 120 (delta 12), reused 64 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 1.64 MiB | 38.13 MiB/s, done.
Resolving deltas: 100% (12/12), done.
mamba3.py found
Patched: mamba3_step_fn stubbed
Mamba3: <class 'mamba_ssm.modules.mamba3.Mamba3'>
OK


In [13]:
# Cell 3: Inspect Mamba3 API
import inspect
from mamba_ssm import Mamba, Mamba2, Mamba3

for name, cls in [("Mamba (v1)", Mamba), ("Mamba2 (v2)", Mamba2), ("Mamba3 (v3)", Mamba3)]:
    sig = inspect.signature(cls.__init__)
    print(f"=== {name} ===")
    for pname, param in sig.parameters.items():
        if pname == "self":
            continue
        default = param.default if param.default != inspect.Parameter.empty else "REQUIRED"
        print(f"  {pname}: {default}")
    print()


=== Mamba (v1) ===
  d_model: REQUIRED
  d_state: 16
  d_conv: 4
  expand: 2
  dt_rank: auto
  dt_min: 0.001
  dt_max: 0.1
  dt_init: random
  dt_scale: 1.0
  dt_init_floor: 0.0001
  conv_bias: True
  bias: False
  use_fast_path: True
  layer_idx: None
  device: None
  dtype: None

=== Mamba2 (v2) ===
  d_model: REQUIRED
  d_state: 128
  d_conv: 4
  conv_init: None
  expand: 2
  headdim: 64
  d_ssm: None
  ngroups: 1
  A_init_range: (1, 16)
  D_has_hdim: False
  rmsnorm: True
  norm_before_gate: False
  dt_min: 0.001
  dt_max: 0.1
  dt_init_floor: 0.0001
  dt_limit: (0.0, inf)
  bias: False
  conv_bias: True
  chunk_size: 256
  use_mem_eff_path: True
  layer_idx: None
  process_group: None
  sequence_parallel: True
  device: None
  dtype: None

=== Mamba3 (v3) ===
  d_model: REQUIRED
  d_state: 128
  expand: 2
  headdim: 64
  ngroups: 1
  rope_fraction: 0.5
  dt_min: 0.001
  dt_max: 0.1
  dt_init_floor: 0.0001
  A_floor: 0.0001
  is_outproj_norm: False
  is_mimo: False
  mimo_rank: 4
 

In [14]:
# Cell 4: Forward pass smoke test
import torch
from mamba_ssm import Mamba3

device = torch.device('cuda')
torch.cuda.reset_peak_memory_stats()

# Test with TextMamba3D-compatible dimensions
# Stage dims: 48, 96, 192, 384 (embed_dim=48, 4 stages)
test_configs = [
    {'d_model': 48,  'name': 'Stage 0 (48)'},
    {'d_model': 96,  'name': 'Stage 1 (96)'},
    {'d_model': 192, 'name': 'Stage 2 (192)'},
    {'d_model': 384, 'name': 'Stage 3 (384)'},
]

# Sequence lengths matching 3D cross-scan at each stage
# patch=4, img=128 -> spatial: 32,16,8,4
seq_lens = [32768, 4096, 512, 64]  # 32^3, 16^3, 8^3, 4^3

print('Forward pass test:')
for cfg, seq_len in zip(test_configs, seq_lens):
    try:
        # Try creating Mamba3 with d_state=16 (same as V4.5/V5.0)
        ssm = Mamba3(
            d_model=cfg['d_model'],
            d_state=16,
            expand=2,
        ).to(device)
        
        x = torch.randn(1, seq_len, cfg['d_model'], device=device)
        with torch.cuda.amp.autocast():
            out = ssm(x)
        
        assert out.shape == x.shape, f'Shape mismatch: {out.shape} != {x.shape}'
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f'  {cfg["name"]}: seq={seq_len} -> {out.shape} OK (peak {peak:.2f} GB)')
        
        del ssm, x, out
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    except Exception as e:
        print(f'  {cfg["name"]}: FAILED - {type(e).__name__}: {e}')

print('Forward pass test complete')

Forward pass test:
  Stage 0 (48): FAILED - AssertionError: 


/tmp/ipykernel_1958/2108411055.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Stage 1 (96): seq=4096 -> torch.Size([1, 4096, 96]) OK (peak 0.27 GB)


/tmp/ipykernel_1958/2108411055.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Stage 2 (192): seq=512 -> torch.Size([1, 512, 192]) OK (peak 0.01 GB)


/tmp/ipykernel_1958/2108411055.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Stage 3 (384): seq=64 -> torch.Size([1, 64, 384]) OK (peak 0.01 GB)
Forward pass test complete


In [15]:
# Cell 5: Backward pass test (this is where Triton kernels may crash)
import torch
from mamba_ssm import Mamba3

device = torch.device('cuda')

print('Backward pass test (Triton kernel stability):')
for d_model, seq_len, name in [
    (48, 512, 'Small (48, L=512)'),
    (96, 512, 'Medium (96, L=512)'),
    (192, 512, 'Large (192, L=512)'),
    (48, 4096, 'Long seq (48, L=4096)'),
    (96, 4096, 'Long seq (96, L=4096)'),
]:
    try:
        ssm = Mamba3(d_model=d_model, d_state=16, expand=2).to(device)
        x = torch.randn(2, seq_len, d_model, device=device, requires_grad=True)
        
        with torch.cuda.amp.autocast():
            out = ssm(x)
            loss = out.sum()
        
        loss.backward()  # <-- This is where illegal memory access may occur
        
        grad_norm = x.grad.norm().item()
        assert not torch.isnan(x.grad).any(), 'NaN in gradients!'
        assert not torch.isinf(x.grad).any(), 'Inf in gradients!'
        print(f'  {name}: OK (grad_norm={grad_norm:.4f})')
        
        del ssm, x, out, loss
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f'  {name}: FAILED - {type(e).__name__}: {e}')
        # If CUDA error, try to recover
        if 'CUDA' in str(e) or 'illegal' in str(e).lower():
            print('  -> CUDA error detected. This is the known Triton kernel bug.')
            print('  -> Remaining tests may be unreliable. Restarting runtime recommended.')
            break

print('Backward pass test complete')

Backward pass test (Triton kernel stability):
  Small (48, L=512): FAILED - AssertionError: 


/tmp/ipykernel_1958/3284347159.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Medium (96, L=512): FAILED - CompilationError: at 288:18:
            s_block = tl.where(
                tl.arange(0, CHUNK_SIZE)[None, :] > tl.arange(0, CHUNK_SIZE)[:, None],
                s_block,
                0.0
            )
        else:
            s_block *= causal_decay_mask

        acc_dq = tl.dot(tl.trans(s_block).to(k_block.dtype), k_block)  # (CHUNK_SIZE, HEADDIM_QK)

        # Inter-chunk: gradient through states from previous chunks
        acc_dq += tl.dot(do_block, ssm_states_block) * exp_da_cs[:, None]
                  ^
Both operands must be same dtype. Got fp16 and bf16
  Large (192, L=512): FAILED - CompilationError: at 288:18:
            s_block = tl.where(
                tl.arange(0, CHUNK_SIZE)[None, :] > tl.arange(0, CHUNK_SIZE)[:, None],
                s_block,
                0.0
            )
        else:
            s_block *= causal_decay_mask

        acc_dq = tl.dot(tl.trans(s_block).to(k_block.dtype), k_block)  # (CHUNK_SIZE, HEADDIM_QK)



/tmp/ipykernel_1958/3284347159.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1958/3284347159.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [19]:
# Cell 5b: Backward pass - pure bf16 (kernel uses bf16 internally)

import torch
from mamba_ssm import Mamba3

device = torch.device("cuda")

print("Backward pass test (explicit bf16):")
for d_model, seq_len, name in [
    (96, 512, "Medium (96, L=512)"),
    (192, 512, "Large (192, L=512)"),
    (96, 4096, "Long seq (96, L=4096)"),
]:
    try:
        ssm = Mamba3(d_model=d_model, d_state=16, expand=2).to(device=device, dtype=torch.bfloat16)
        x = torch.randn(2, seq_len, d_model, device=device, dtype=torch.bfloat16, requires_grad=True)
        
        out = ssm(x)
        loss = out.sum()
        loss.backward()
        
        grad_norm = x.grad.norm().item()
        has_nan = torch.isnan(x.grad).any().item()
        print(f"  {name}: OK (grad_norm={grad_norm:.4f}, nan={has_nan})")
        
        del ssm, x, out, loss
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"  {name}: FAILED - {type(e).__name__}: {e}")

print("Done")


Backward pass test (explicit bf16):
  Medium (96, L=512): OK (grad_norm=218.0000, nan=False)
  Large (192, L=512): OK (grad_norm=224.0000, nan=False)
  Long seq (96, L=4096): OK (grad_norm=780.0000, nan=False)
Done


In [20]:
# Cell 6: CrossScan integration test (bf16)

import os, sys, torch
REPO_DIR = "/content/TextMamba3D"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/PlutoLei/TextMamba3D.git {REPO_DIR}
os.chdir(REPO_DIR)
if "." not in sys.path:
    sys.path.insert(0, ".")

from models.mamba_block import CrossScanBiMamba3DBlock, MAMBA3_AVAILABLE
print(f"MAMBA3_AVAILABLE: {MAMBA3_AVAILABLE}")

device = torch.device("cuda")

print("CrossScanBiMamba3DBlock + Mamba3 (bf16):")
for dim, spatial, name in [
    (96, (16,16,16), "Stage 1 (96, 16^3)"),
    (192, (8,8,8), "Stage 2 (192, 8^3)"),
    (384, (4,4,4), "Stage 3 (384, 4^3)"),
]:
    try:
        seq_len = spatial[0] * spatial[1] * spatial[2]
        block = CrossScanBiMamba3DBlock(
            dim=dim, spatial_dims=spatial, d_state=16, expand=2, use_mamba3=True,
        ).to(device=device, dtype=torch.bfloat16)
        
        x = torch.randn(2, seq_len, dim, device=device, dtype=torch.bfloat16, requires_grad=True)
        out = block(x)
        loss = out.sum()
        loss.backward()
        
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  {name}: OK (grad={x.grad.norm().item():.2f}, peak={peak:.2f}GB)")
        
        del block, x, out, loss
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    except Exception as e:
        print(f"  {name}: FAILED - {type(e).__name__}: {e}")

print("Done")


MAMBA3_AVAILABLE: True
CrossScanBiMamba3DBlock + Mamba3 (bf16):
  Stage 1 (96, 16^3): OK (grad=964.00, peak=2.05GB)
  Stage 2 (192, 8^3): OK (grad=476.00, peak=1.15GB)
  Stage 3 (384, 4^3): OK (grad=227.00, peak=1.40GB)
Done


In [21]:
# Cell 7: Full TextMamba3D + Mamba3 smoke test (bf16)

import os, sys, torch
REPO_DIR = "/content/TextMamba3D"
os.chdir(REPO_DIR)
if "." not in sys.path:
    sys.path.insert(0, ".")

from models.textmamba3d import TextMamba3D

device = torch.device("cuda")
torch.cuda.reset_peak_memory_stats()

print("Full TextMamba3D + Mamba3 (bf16):")
try:
    model = TextMamba3D(
        img_size=(128, 128, 128),
        in_channels=4,
        out_channels=4,
        embed_dim=48,
        depths=[2, 2, 2, 2],
        d_state=16,
        text_embed_dim=256,
        text_max_len=192,
        use_mamba3=True,
        fusion_type="seqca",
        deep_supervision=True,
    ).to(device=device, dtype=torch.bfloat16)

    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Params: {params:,}")

    img = torch.randn(1, 4, 128, 128, 128, device=device, dtype=torch.bfloat16)
    text_ids = torch.randint(0, 30000, (1, 64), device=device)
    attn_mask = torch.ones(1, 64, device=device)

    out = model(img, text_ids, attn_mask)
    if isinstance(out, (list, tuple)):
        loss = out[0].sum()
    else:
        loss = out.sum()
    loss.backward()

    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    out_shape = out[0].shape if isinstance(out, (list, tuple)) else out.shape
    print(f"  Output: {out_shape}")
    print(f"  Peak GPU: {peak:.1f} / {total:.0f} GB")
    print(f"  PASS")

    del model, img, text_ids, attn_mask, out, loss
    torch.cuda.empty_cache()

except Exception as e:
    print(f"  FAILED: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()


Full TextMamba3D + Mamba3 (bf16):


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Params: 21,958,952
  Output: torch.Size([1, 4, 128, 128, 128])
  Peak GPU: 1.7 / 39 GB
  PASS


## Results Summary

Fill in after running:

| Test | Status | Notes |
|------|--------|-------|
| Install | | |
| Forward pass | | |
| Backward pass | | |
| CrossScan integration | | |
| Full model | | |

If all pass: proceed to create `TextMamba3D_A100_V5.1.ipynb` (real Mamba-3 training)  
If backward fails: report specific error for upstream fix